In [ ]:
pip install ftfy

In [ ]:
import pandas as pd
import requests
import numpy as np
import seaborn as sns
import matplotlib
import matplotlib.pyplot as plt
import json
from bs4 import BeautifulSoup
import requests
import math
import scipy.stats as stats
import random
import datetime
import unicodedata
import re
import string
from geopy.geocoders import Nominatim
from urllib.parse import urlparse

In [ ]:
# Affichage complet colonnes, observations, etc.
pd.set_option('display.max_columns', None)  # Affiche toutes les colonnes
pd.set_option('display.max_rows', None)     # (optionnel) Affiche toutes les lignes si nécessaire
pd.set_option('display.width', None)        # Pas de coupure de ligne
pd.set_option('display.max_colwidth', None) # Affiche tout le contenu des cellules

In [ ]:
def corriger_et_nettoyer_texte(texte):
    if not isinstance(texte, str):
        return texte  # Ne traite que les chaînes

    # 1. Correction d'encodage
    try:
        texte = texte.encode('raw_unicode_escape').decode('utf-8')
    except Exception:
        try:
            texte = texte.encode('latin1', errors='ignore').decode('utf-8', errors='ignore')
        except Exception:
            pass  # On garde le texte tel quel

    # 2. Nettoyage des artefacts type Word/Excel
    texte = texte.replace('_x000D_', ' ')

    # 2.1. Suppression des séquences encodées de type _xHHHH_
    texte = re.sub(r'_x[0-9A-Fa-f]{4}_', '', texte)

    # 3. Suppression de caractères invisibles ou bizarres
    texte = re.sub(r'[\u00A0\u200B\u2028\u2029\r\n]+', ' ', texte)

    # 4. Nettoyage des espaces doubles
    texte = re.sub(r'\s+', ' ', texte)

    return texte.strip()

## Et tu l’appliques comme ceci :

## Modifier
## df["domaine"] = df["domaine"].apply(corriger_et_nettoyer_texte)

In [ ]:
# Fonction traitement des faux NaN
#je crée une liste de faux nul, ce qui permettra par la suite de pouvoir les traiter
faux_nans = ['NaN', 'nan', 'None', 'NULL', 'null', 'na', '-', '--', '', ' ', '\\n']
def detecter_faux_nan(df, valeurs_fausses=faux_nans):   #on crée la fonction afin d'identifier les nans
    faux_nan_counts = {}

    for col in df.columns:
        # On travaille sur des chaînes nettoyées (strip) et converties en str
        faux = df[col].astype(str).str.strip().isin(valeurs_fausses)
        nb_faux = faux.sum()
        if nb_faux > 0:
            faux_nan_counts[col] = nb_faux

    return pd.Series(faux_nan_counts).sort_values(ascending=False)

In [ ]:
# fonction pour nettoyer les codes des sites web
def extraire_domaine(url):
    if not isinstance(url, str):
        return url
    try:
        parsed = urlparse(url if '://' in url else 'http://' + url)  # ajoute http:// si absent
        domaine = parsed.netloc
        return domaine if domaine else url
    except:
        return url

In [ ]:
## Lecture de la table domaine
df_domaine = pd.read_excel('/content/Principal_Domaines.xlsx')

In [ ]:
# On applique la fonction à notre df pour toutes les colonnes qui sont des str afin de remplacer les faux "nan" par les vrais "nan"
df_domaine = df_domaine.apply(lambda col: col.replace(faux_nans, np.nan) if col.dtype == "object" else col)

In [ ]:
df_domaine_maj = df_domaine[['id_domaine', # clé de la table, changer le type de int à Object
                             'id_proprio', # clé secondaire vers la table proprio, peu exploitable
                             'id_region_domaine', # clé secondaire, exploitable
                             'id_picto_bio', # clé secondaire vers la table picto-bio, peu exploitable
                             'id_picto_valorisation', # clé secondaire vers la table picto-valo, peu exploitable
                             'nom_du_domaine', # exploitable, modifier les noms en majuscules vers minuscule sur power query
                             'index_cle_de_tri', # repetition du nom du domaine avec des null, à drop
                             'note_domaine', # exploitable
                             'description_domaine', # exploitable, modifier les noms en majuscules vers minuscule sur power query
                             'adresse_dom', # exploitable sous condition, modifier les noms en majuscules vers minuscule sur power query
                             'cp_dom', # exploitable sous condition
                             'ville_dom', # exploitable, modifier les noms en majuscules vers minuscule sur power query
                             'telephone_dom',# exploitable => modifier numero telephone au bon format (manque 0 pour la fr)
                             'fax_dom', # exploitable
                             'mail_dom', # exploitable
                             'site_dom', # exploitable ; + reperer les erreurs type mail à l'interrieur
                             'visite_dom',
                             'created',
                             'published']]

In [ ]:
# Boucle pour corriger les erreur de texte sur les colonnes
colonne_a_convertir = ['nom_du_domaine', 'index_cle_de_tri', 'description_domaine', 'adresse_dom', 'ville_dom','visite_dom']
for x in colonne_a_convertir:
    df_domaine_maj[x] = df_domaine_maj[x].apply(corriger_et_nettoyer_texte)

In [ ]:
# Mettre les colonnes au bon Dtype
df_domaine_maj['id_domaine'] = df_domaine_maj['id_domaine'].astype(str)
df_domaine_maj['id_proprio'] = df_domaine_maj['id_proprio'].astype(str)
df_domaine_maj['id_region_domaine'] = df_domaine_maj['id_region_domaine'].astype(str)
df_domaine_maj['id_picto_bio'] = df_domaine_maj['id_picto_bio'].astype(str)
df_domaine_maj['id_picto_valorisation'] = df_domaine_maj['id_picto_valorisation'].astype(str)
df_domaine_maj['cp_dom'] = df_domaine_maj['cp_dom'].astype(str)
df_domaine_maj['note_domaine'] = df_domaine_maj['note_domaine'].astype(str)

## Mettre tout ce qui est faux nan en vrai nan
df_domaine_maj = df_domaine_maj.replace('nan', np.nan)

In [ ]:
# droper les lignes inutiles
df_domaine_maj = df_domaine_maj.dropna(subset=['nom_du_domaine']) # retire les lignes vides dans "nom du domaine"
df_domaine_maj = df_domaine_maj[df_domaine_maj['nom_du_domaine'] != 'test'] # retire la ligne test
# remplacer des valeurs
df_domaine_maj['note_domaine'] = df_domaine_maj['note_domaine'].replace(-1, 1)
df_domaine_maj['note_domaine'] = df_domaine_maj['note_domaine'].replace(0, np.nan)
df_domaine_maj['description_domaine'] = df_domaine_maj['description_domaine'].replace('Commentaire à faire', np.nan)
df_domaine_maj['description_domaine'] = df_domaine_maj['description_domaine'].replace('jwhbdklhfljds', np.nan)
df_domaine_maj['description_domaine'] = df_domaine_maj['description_domaine'].replace('jkfjkgfjkhsdmkjgqksfjqsrbjk', np.nan)
df_domaine_maj['description_domaine'] = df_domaine_maj['description_domaine'].replace('jwhbdklhfljds', np.nan)
df_domaine_maj['description_domaine'] = df_domaine_maj['description_domaine'].replace('Non', np.nan)
df_domaine_maj['description_domaine'] = df_domaine_maj['description_domaine'].replace('non retenu', np.nan)
df_domaine_maj['description_domaine'] = df_domaine_maj['description_domaine'].replace('non, qualité faible', np.nan)
df_domaine_maj['description_domaine'] = df_domaine_maj['description_domaine'].replace('non, sorti du guide', np.nan)
df_domaine_maj['cp_dom'] = df_domaine_maj['cp_dom'].str.replace('.0', '')

## remplacer les valeurs (ref : boucle pour les mail avec un '@' manquant)
df_domaine_maj['mail_dom'] = df_domaine_maj['mail_dom'].replace('03 80 61 00 64', np.nan)
df_domaine_maj.loc[1796, 'mail_dom'] = df_domaine_maj.loc[1796, 'mail_dom'].replace(']', '@')
df_domaine_maj['mail_dom'] = df_domaine_maj['mail_dom'].replace('contactdomaineka.fr', 'contact@domaineka.fr')
df_domaine_maj.loc[5898, 'telephone_dom'] = df_domaine_maj.loc[5898, 'mail_dom']
df_domaine_maj.loc[5898, 'mail_dom'] = np.nan
df_domaine_maj.loc[8289, 'telephone_dom'] = df_domaine_maj.loc[8289, 'mail_dom']
df_domaine_maj.loc[8289, 'mail_dom'] = np.nan
df_domaine_maj['mail_dom'] = df_domaine_maj['mail_dom'].replace('www.champagne-charlesheston.com', 'contact@champagne-charlesheston.com')
df_domaine_maj['site_dom'] = df_domaine_maj['site_dom'].replace('contact@champagne-charlesheston.com', 'www.champagne-charlesheston.com')
# site
df_domaine_maj['site_dom'] = df_domaine_maj['site_dom'].replace('We.domainedesperches.wine', 'domainedesperches.wine')

In [ ]:
## Boucle pour trouver les mails avec un '@' manquant
invalides = df_domaine_maj[
    df_domaine_maj['mail_dom'].notnull() &
    ~df_domaine_maj['mail_dom'].apply(lambda x: isinstance(x, str) and '@' in x)
]
invalides

In [ ]:
# Boucle pour extraire le site web au bon format
df_domaine_maj['site_dom_nettoyer'] = df_domaine_maj['site_dom'].apply(extraire_domaine)

In [ ]:
df_domaine_maj[['site_dom','site_dom_nettoyer']][df_domaine_maj['site_dom_nettoyer'].notna()]

In [ ]:
## 182 Lignes avec 4 caractères => nettoyage codes postaux
df_domaine_maj[["id_domaine","adresse_dom",'cp_dom']][df_domaine_maj['adresse_dom'].notna() & df_domaine_maj['cp_dom'].str.len().eq(4)].sort_values(by='cp_dom', ascending=True)

In [ ]:
# modification format tel et fax

# Fonction pour ajouter un espace entre chaque chiffre
def format_telephone(numero):
    if pd.notna(numero):  # Vérifier si le numéro n'est pas NaN
        return " ".join(str(numero))  # Convertir en string et ajouter des espaces
    return numero

# Appliquer la transformation aux colonnes 'telephone_dom' et 'fax_dom'
df_domaine['telephone_dom'] = df_domaine['telephone_dom'].apply(format_telephone)
df_domaine['fax_dom'] = df_domaine['fax_dom'].apply(format_telephone)

In [ ]:
df_domaine_maj.info()